# Autoresearch Quantum — Control Room

This interactive dashboard lets you run encoded magic-state experiments with different parameter settings
and instantly visualize the results. Use it alongside the three Track notebooks:

- **Track A** (Physics): `track_a_physics.ipynb`
- **Track B** (Engineering): `track_b_engineering.ipynb`
- **Track C** (Search): `track_c_search.ipynb`

Each track will suggest "Dashboard Exercises" that bring you back here for hands-on exploration.

In [ ]:
%matplotlib inline

import tempfile
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from autoresearch_quantum.codes.four_two_two import (
    build_preparation_circuit, build_encoder, apply_magic_seed,
    encoded_magic_statevector, STABILIZERS, MEASUREMENT_OPERATORS, DATA_QUBITS,
)
from autoresearch_quantum.experiments.encoded_magic_state import build_circuit_bundle
from autoresearch_quantum.models import (
    ExperimentSpec, RungConfig, EvaluationMetrics, TierResult,
    QualityWeights, CostWeights, ScoreConfig, SearchSpaceConfig,
    TierPolicyConfig, HardwareConfig,
)
from autoresearch_quantum.execution.local import LocalCheapExecutor
from autoresearch_quantum.execution.backends import resolve_backend, backend_metadata
from autoresearch_quantum.execution.transpile import (
    transpile_circuits, count_two_qubit_gates, runtime_estimate, circuit_metadata,
)
from autoresearch_quantum.scoring.score import score_metrics
from autoresearch_quantum.config import load_rung_config

print("All imports successful.")

In [ ]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, checkpoint_summary
tracker = LearningTracker("plan_c_dashboard")
print("Learning tracker active.")

## Setup: Load Base Configuration

We load the rung-1 configuration as a baseline. The dashboard widgets will override individual parameters.

In [ ]:
# Load rung-1 config as our baseline
rung_config = load_rung_config("../../configs/rungs/rung1.yaml")
executor = LocalCheapExecutor()

# Storage for comparison runs
run_history = []

print(f"Loaded config: {rung_config.name}")
print(f"Objective: {rung_config.objective}")

In [ ]:
quiz(tracker, "q1_baseline",
    question="Why does the dashboard start from a rung-1 config as baseline?",
    options=[
        "It is the only config that exists",
        "It provides sensible defaults that widgets then override one at a time",
        "Higher rungs require IBM hardware access",
    ],
    correct=1, section="1. Setup", bloom="understand",
    explanation="The rung-1 config defines the full parameter space and bootstrap incumbent. Widgets let you explore variations.")

## Interactive Dashboard

Adjust the widgets below and click **Run Experiment** to execute a single experiment with the chosen
parameters. Click **Compare** to overlay a second run on the same plots (in orange) for side-by-side
comparison.

**Tip**: Try changing `seed_style` among `h_p`, `ry_rz`, `u_magic` — the witness should stay roughly
the same (they are equivalent preparations up to global phase).

Now run three experiments using the dashboard above:
1. Set seed_style to `h_p`, click **Run Experiment**
2. Change to `ry_rz`, click **Compare**
3. Change to `u_magic`, click **Compare**

Look at the "Quality Metrics" panel. Record what you see, then verify your prediction below.

In [ ]:
predict_choice(tracker, "q2_verification_effect",
    question="What happens to acceptance rate if you set verification to 'none'?",
    options=[
        "Acceptance rate drops because there are no checks",
        "Acceptance rate goes to 100% because no shots are filtered",
        "No change \u2014 verification doesn't affect acceptance",
    ],
    correct=1, section="2. Exploration", bloom="apply",
    explanation="With verification='none', there are no syndrome checks, so ALL shots are accepted (100%). But quality may be lower.")

In [ ]:
reflect(tracker, "q3_tradeoff",
    question="After trying several parameter combinations, what tension do you notice between quality and acceptance?",
    section="2. Exploration", bloom="analyze",
    model_answer="Stricter verification improves quality by filtering errors, but reduces acceptance rate. The score balances this: score = quality \u00d7 acceptance / cost.")
checkpoint_summary(tracker, "2. Exploration")

In [ ]:
# ── Widget definitions ─────────────────────────────────────────────
w_seed = widgets.Dropdown(
    options=["h_p", "ry_rz", "u_magic"],
    value="h_p",
    description="Seed style:",
)
w_encoder = widgets.Dropdown(
    options=["cx_chain", "cz_compiled"],
    value="cx_chain",
    description="Encoder:",
)
w_verification = widgets.Dropdown(
    options=["both", "z_only", "x_only", "none"],
    value="both",
    description="Verification:",
)
w_postselection = widgets.Dropdown(
    options=["all_measured", "z_only", "x_only", "none"],
    value="all_measured",
    description="Postselect:",
)
w_opt_level = widgets.IntSlider(
    value=2, min=1, max=3, step=1,
    description="Opt level:",
)
w_shots = widgets.IntSlider(
    value=256, min=64, max=1024, step=64,
    description="Shots:",
)
w_backend = widgets.Dropdown(
    options=["fake_brisbane"],
    value="fake_brisbane",
    description="Backend:",
)

btn_run = widgets.Button(description="Run Experiment", button_style="primary")
btn_compare = widgets.Button(description="Compare", button_style="warning")
btn_clear = widgets.Button(description="Clear History", button_style="danger")

out = widgets.Output()
out_text = widgets.Output()


def _build_spec():
    """Build an ExperimentSpec from the current widget values."""
    return ExperimentSpec(
        rung=1,
        seed_style=w_seed.value,
        encoder_style=w_encoder.value,
        verification=w_verification.value,
        postselection=w_postselection.value,
        ancilla_strategy="dedicated_pair",
        optimization_level=w_opt_level.value,
        layout_method="sabre",
        routing_method="sabre",
        approximation_degree=1.0,
        target_backend=w_backend.value,
        noise_backend=w_backend.value,
        shots=w_shots.value,
        repeats=1,
    )


def _run_and_collect():
    """Run experiment, return (spec, tier_result, prep_circuit, bundle)."""
    spec = _build_spec()
    result = executor.evaluate(spec, rung_config)
    bundle = build_circuit_bundle(spec)
    backend = resolve_backend(spec.target_backend, rung_config.hardware)
    transpiled = transpile_circuits([bundle.acceptance], spec, backend)[0]
    return spec, result, bundle.prep, transpiled


def _label(spec):
    return f"{spec.seed_style}/{spec.encoder_style}/opt{spec.optimization_level}"


def _plot_all(clear=True):
    """Draw the 2x2 dashboard from run_history."""
    with out:
        if clear:
            clear_output(wait=True)

        if not run_history:
            print("No runs yet. Click 'Run Experiment'.")
            return

        colors = plt.cm.tab10.colors
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("Autoresearch Quantum — Control Room", fontsize=14, fontweight="bold")

        # ── Top-left: Circuit diagram (latest run only) ──
        ax_circ = axes[0, 0]
        ax_circ.set_axis_off()
        latest_prep = run_history[-1][2]
        latest_label = _label(run_history[-1][0])
        ax_circ.set_title(f"Preparation Circuit ({latest_label})", fontsize=11)
        # Draw circuit as text
        circ_text = latest_prep.draw(output="text").__str__()
        ax_circ.text(
            0.02, 0.95, circ_text,
            transform=ax_circ.transAxes, fontsize=7, verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8),
        )

        # ── Top-right: Histogram of acceptance counts ──
        ax_hist = axes[0, 1]
        ax_hist.set_title("Measurement Outcome Counts (acceptance circuit)", fontsize=11)
        bar_width = 0.8 / max(len(run_history), 1)
        for i, (spec, result, _, _) in enumerate(run_history):
            counts = result.counts_summary.get("acceptance", {}).get("latest", {})
            raw = counts.get("raw_data_counts", {})
            if raw:
                keys = sorted(raw.keys())
                vals = [raw[k] for k in keys]
                x = np.arange(len(keys))
                ax_hist.bar(
                    x + i * bar_width, vals, bar_width,
                    label=_label(spec), color=colors[i % len(colors)], alpha=0.8,
                )
                if i == len(run_history) - 1:
                    ax_hist.set_xticks(x + bar_width * (len(run_history) - 1) / 2)
                    ax_hist.set_xticklabels(keys, rotation=90, fontsize=6)
        ax_hist.set_ylabel("Counts")
        ax_hist.legend(fontsize=8)

        # ── Bottom-left: Quality metrics bar chart ──
        ax_metrics = axes[1, 0]
        ax_metrics.set_title("Quality Metrics", fontsize=11)
        metric_names = ["acceptance_rate", "witness", "score"]
        x = np.arange(len(metric_names))
        bar_w = 0.8 / max(len(run_history), 1)
        for i, (spec, result, _, _) in enumerate(run_history):
            m = result.metrics
            vals = [
                m.acceptance_rate,
                m.logical_magic_witness or 0.0,
                result.score,
            ]
            ax_metrics.barh(
                x + i * bar_w, vals, bar_w,
                label=_label(spec), color=colors[i % len(colors)], alpha=0.8,
            )
        ax_metrics.set_yticks(x + bar_w * (len(run_history) - 1) / 2)
        ax_metrics.set_yticklabels(metric_names)
        ax_metrics.set_xlim(0, 1.0)
        ax_metrics.legend(fontsize=8)

        # ── Bottom-right: Transpile cost stats ──
        ax_cost = axes[1, 1]
        ax_cost.set_title("Transpile & Cost Stats", fontsize=11)
        cost_names = ["2Q gates", "depth", "total_cost"]
        x = np.arange(len(cost_names))
        bar_w = 0.8 / max(len(run_history), 1)
        for i, (spec, result, _, _) in enumerate(run_history):
            m = result.metrics
            vals = [m.two_qubit_count, m.depth, m.total_cost]
            ax_cost.bar(
                x + i * bar_w, vals, bar_w,
                label=_label(spec), color=colors[i % len(colors)], alpha=0.8,
            )
        ax_cost.set_xticks(x + bar_w * (len(run_history) - 1) / 2)
        ax_cost.set_xticklabels(cost_names)
        ax_cost.legend(fontsize=8)

        plt.tight_layout()
        plt.show()


def _print_summary():
    """Print textual summary of the latest run."""
    with out_text:
        clear_output(wait=True)
        if not run_history:
            return
        spec, result, _, _ = run_history[-1]
        m = result.metrics
        print("=" * 60)
        print(f"  Run #{len(run_history)}:  {_label(spec)}")
        print("=" * 60)
        print(f"  Score:               {result.score:.4f}")
        print(f"  Quality estimate:    {result.quality_estimate:.4f}")
        print(f"  Acceptance rate:     {m.acceptance_rate:.4f}")
        print(f"  Magic witness:       {m.logical_magic_witness:.4f}" if m.logical_magic_witness else "  Magic witness:       N/A")
        print(f"  Ideal fidelity:      {m.ideal_encoded_fidelity:.4f}" if m.ideal_encoded_fidelity else "  Ideal fidelity:      N/A")
        print(f"  Noisy fidelity:      {m.noisy_encoded_fidelity:.4f}" if m.noisy_encoded_fidelity else "  Noisy fidelity:      N/A")
        print(f"  Spectator Z:         {m.spectator_logical_z:.4f}" if m.spectator_logical_z else "  Spectator Z:         N/A")
        print(f"  Stability score:     {m.stability_score:.4f}" if m.stability_score else "  Stability score:     N/A")
        print(f"  Two-qubit gates:     {m.two_qubit_count}")
        print(f"  Circuit depth:       {m.depth}")
        print(f"  Total cost:          {m.total_cost:.4f}")
        print(f"  Failure mode:        {m.dominant_failure_mode}")
        print("=" * 60)


def on_run(btn):
    """Run a fresh experiment (replaces history)."""
    run_history.clear()
    spec, result, prep, transpiled = _run_and_collect()
    run_history.append((spec, result, prep, transpiled))
    _plot_all(clear=True)
    _print_summary()


def on_compare(btn):
    """Run experiment and overlay on existing plots."""
    spec, result, prep, transpiled = _run_and_collect()
    run_history.append((spec, result, prep, transpiled))
    _plot_all(clear=True)
    _print_summary()


def on_clear(btn):
    """Clear all history."""
    run_history.clear()
    with out:
        clear_output(wait=True)
        print("History cleared.")
    with out_text:
        clear_output(wait=True)


btn_run.on_click(on_run)
btn_compare.on_click(on_compare)
btn_clear.on_click(on_clear)

# ── Layout ──
controls_left = widgets.VBox([w_seed, w_encoder, w_verification, w_postselection])
controls_right = widgets.VBox([w_opt_level, w_shots, w_backend])
buttons = widgets.HBox([btn_run, btn_compare, btn_clear])
control_panel = widgets.HBox([controls_left, controls_right])

display(widgets.VBox([
    widgets.HTML("<h3>Experiment Controls</h3>"),
    control_panel,
    buttons,
    out,
    out_text,
]))

**Dashboard Exercise:**
1. Set verification=`both`, postselection=`all_measured`. Click **Run Experiment**.
2. Set verification=`none`, postselection=`none`. Click **Compare**.

Compare the acceptance rates and witness values.

## How to Use This Dashboard

1. **Single run**: Adjust parameters, click **Run Experiment**. All four panels update.
2. **Compare**: Change a parameter and click **Compare**. The new result overlays in a different color.
3. **Clear**: Click **Clear History** to reset.

### Suggested Explorations

| Experiment | What to change | What to watch |
|---|---|---|
| Seed equivalence | `seed_style`: h_p, ry_rz, u_magic | Witness should be stable |
| Encoder comparison | `encoder_style`: cx_chain vs cz_compiled | Gate count differs |
| Optimization impact | `optimization_level`: 1 vs 3 | Gate count drops, score rises |
| Verification effect | `verification`: both vs none | Acceptance rate changes |
| Shot noise | `shots`: 64 vs 512 | Metrics variance decreases |

**Dashboard Exercise:**
1. Set opt_level=1. Click **Run Experiment**.
2. Set opt_level=3. Click **Compare**.
3. Look at BOTH the quality metrics AND the cost stats panels.

### Free Response: Dashboard Synthesis

Now that you have explored seeds, verification, and optimization levels, reflect on the relationships.

---
## Learning Dashboard

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")